# Tennis ball tracking — full pipeline

| Step | What | Output |
|---|---|---|
| 1 | Frame-diff vs background subtraction | Figure |
| 2 | Candidate detector (full frame, no mask) | Per-frame blob lists (cached) |
| 3 | Detection statistics | Histograms, scatter, time-series |
| 4 | Pipeline debug frames | 6-panel PNGs |
| 5 | Kalman filter tracker | Smooth trajectory |
| 6 | Trajectory analysis | x(t), y(t), speed, heatmap |
| 7 | Activity detection | Segments vs audio |
| 8 | Videos | framediff + tracking (10 min each) |

In [ ]:
from pathlib import Path as _P

_VIDEO_NAME = "Raja vs Wijemanne.mp4"
_ROOT = next(
    (p for p in [_P.cwd(), _P.cwd().parent, _P.cwd().parent.parent]
     if (p / "videosandaudio" / _VIDEO_NAME).exists()), None)
if _ROOT is None:
    raise FileNotFoundError(f"Cannot find '{_VIDEO_NAME}' near cwd.")

VIDEO_PATH    = str(_ROOT / "videosandaudio" / _VIDEO_NAME)
BG_CACHE      = str(_ROOT / "svdanalysis" / "backgrounds" / "clean_background_median.png")
SCAN_CACHE    = str(_ROOT / "svdanalysis" / "framediff_scan_5min.pkl")
GENVIDEOS_DIR = str(_ROOT / "claude" / "genvideos")
DEBUG_DIR     = str(_ROOT / "claude" / "genvideos" / "debug_frames")

FRAME_W, FRAME_H = 320, 180
VID_MINUTES       = 5
DIFF_FRAMES       = 2
DIFF_AMP          = 6.0
DIFF_THRESH       = 10
MIN_AREA          = 2
MAX_AREA          = 80
MAX_ASPECT        = 3.0
DEBUG_EVERY       = 125

COURT_POLY_PTS = [[116, 50], [193, 52], [293, 118], [16, 118]]

KF_PROC_POS = 1.0
KF_PROC_VEL = 8.0
KF_MEAS     = 3.0
KF_GATE     = 4.0
KF_MAX_MISS = 10

ACT_WIN_S  = 2.0
ACT_THRESH = 0.08

In [ ]:
import cv2, numpy as np, os, pickle
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

os.makedirs(GENVIDEOS_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)
os.makedirs(str(_ROOT / "svdanalysis"), exist_ok=True)

def video_info(path):
    cap = cv2.VideoCapture(path)
    info = {"fps": cap.get(cv2.CAP_PROP_FPS),
            "n_frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            "width":  int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}
    info["duration_s"]   = info["n_frames"] / info["fps"]
    info["duration_min"] = info["duration_s"] / 60
    cap.release()
    return info

info = video_info(VIDEO_PATH)
FPS  = info["fps"]
print(f"Video: {info['width']}x{info['height']}  {FPS:.1f} fps  "
      f"{info['n_frames']:,} frames  {info['duration_min']:.1f} min")

clean_bg = cv2.imread(BG_CACHE, cv2.IMREAD_GRAYSCALE).astype(np.float32)
print(f"Background: shape={clean_bg.shape}  mean={clean_bg.mean():.1f}")

COURT_POLY = np.array(COURT_POLY_PTS, dtype=np.int32)
# court mask kept only for visualisation overlays — NOT used in detection
court_mask = np.zeros((FRAME_H, FRAME_W), dtype=np.uint8)
cv2.fillPoly(court_mask, [COURT_POLY], 255)
print("Setup done.")

---
## Step 1 — Frame-diff vs background subtraction

Frame-diff is clean: static objects (lines, stands) are black; only motion is bright.
No court mask needed — the ball is a tiny isolated hot dot, easily separable.

In [ ]:
sample_t  = [20, 80, 230]
sample_lb = ["dead (0:20)", "active (1:20)", "rally (3:50)"]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("Frame-diff (right) vs background subtraction (middle)\n"
             "Frame-diff: static scene is black, only motion glows",
             fontsize=12, fontweight="bold")

cap = cv2.VideoCapture(VIDEO_PATH)
for row, (t_s, lbl) in enumerate(zip(sample_t, sample_lb)):
    fidx = int(t_s * FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, fidx - DIFF_FRAMES))
    _, f_prev = cap.read()
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ok, f_curr = cap.read()
    if not ok: continue
    g_curr = cv2.resize(cv2.cvtColor(f_curr, cv2.COLOR_BGR2GRAY), (FRAME_W, FRAME_H))
    g_prev = cv2.resize(cv2.cvtColor(f_prev, cv2.COLOR_BGR2GRAY), (FRAME_W, FRAME_H))
    bg_d   = cv2.absdiff(g_curr.astype(np.float32), clean_bg).astype(np.uint8)
    fd_d   = cv2.absdiff(g_curr, g_prev)
    def hot(im):
        return cv2.cvtColor(cv2.applyColorMap(
            np.clip(im.astype(float)*DIFF_AMP,0,255).astype(np.uint8),
            cv2.COLORMAP_HOT), cv2.COLOR_BGR2RGB)
    orig = cv2.cvtColor(cv2.resize(f_curr,(FRAME_W,FRAME_H)), cv2.COLOR_BGR2RGB)
    axes[row,0].imshow(orig);      axes[row,0].set_title(f"Original ({lbl})", fontsize=9)
    axes[row,1].imshow(hot(bg_d)); axes[row,1].set_title(f"BG-sub  max={bg_d.max()}", fontsize=9)
    axes[row,2].imshow(hot(fd_d)); axes[row,2].set_title(f"Frame-diff  max={fd_d.max()}", fontsize=9)
    for ax in axes[row]: ax.axis("off")
cap.release()
plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step1_comparison.png"),dpi=110,bbox_inches="tight")
plt.show()
print("Saved step1_comparison.png")

---
## Step 2 — Candidate detector (full frame, no mask)

Ball during serve goes above the court polygon, so we search the whole frame.
The frame-diff is already clean enough — players are large blobs (filtered by MAX_AREA),
ball is a small compact dot that passes the size + compactness score.

In [ ]:
def detect_candidates(g_curr, g_prev,
                      thresh=DIFF_THRESH, min_a=MIN_AREA,
                      max_a=MAX_AREA, max_asp=MAX_ASPECT):
    diff  = cv2.absdiff(g_curr, g_prev)       # full frame — no court mask
    _, bw = cv2.threshold(diff, thresh, 255, cv2.THRESH_BINARY)
    bw    = cv2.morphologyEx(bw, cv2.MORPH_CLOSE,
                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)))
    n, _, stats, centroids = cv2.connectedComponentsWithStats(bw)
    cands = []
    for i in range(1, n):
        a   = stats[i, cv2.CC_STAT_AREA]
        bwi = stats[i, cv2.CC_STAT_WIDTH]
        bhi = stats[i, cv2.CC_STAT_HEIGHT]
        if not (min_a <= a <= max_a):                   continue
        if max(bwi,bhi)/(min(bwi,bhi)+1e-3) > max_asp: continue
        cx, cy = centroids[i]
        comp   = a / (bwi*bhi + 1e-3)
        sscr   = max(1.0 - abs(a-15)/30.0, 0.1)
        cands.append({"x":float(cx),"y":float(cy),
                      "x0":int(stats[i,cv2.CC_STAT_LEFT]),
                      "y0":int(stats[i,cv2.CC_STAT_TOP]),
                      "w":int(bwi),"h":int(bhi),
                      "area":int(a),"score":float(comp*sscr)})
    cands.sort(key=lambda c: c["score"], reverse=True)
    return diff, bw, cands

print("detect_candidates() ready — full frame, no court mask.")

---
## Step 3 — Scan first 5 min (cached)

Delete `svdanalysis/framediff_scan_5min.pkl` to force a re-scan with new settings.

In [ ]:
if os.path.exists(SCAN_CACHE):
    print(f"Loading cached scan: {SCAN_CACHE}")
    with open(SCAN_CACHE, "rb") as fh:
        scan_data = pickle.load(fh)
else:
    N_SCAN = int(VID_MINUTES * 60 * FPS)
    buf    = deque(maxlen=DIFF_FRAMES + 1)
    all_candidates = {}
    sample_frames  = {}

    def hot(im):
        return cv2.cvtColor(cv2.applyColorMap(
            np.clip(im.astype(float)*DIFF_AMP,0,255).astype(np.uint8),
            cv2.COLORMAP_HOT), cv2.COLOR_BGR2RGB)

    cap = cv2.VideoCapture(VIDEO_PATH)
    for fid in range(N_SCAN):
        ok, frame = cap.read()
        if not ok: break
        small = cv2.resize(frame, (FRAME_W, FRAME_H))
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        buf.append((gray, small))

        if len(buf) <= DIFF_FRAMES:
            all_candidates[fid] = []
            continue

        g_curr, s_curr = buf[-1]
        g_prev, s_prev = buf[-(DIFF_FRAMES+1)]
        diff, binary, cands = detect_candidates(g_curr, g_prev)
        all_candidates[fid] = cands

        if fid % DEBUG_EVERY == 0:
            sample_frames[fid] = {
                "orig":   cv2.cvtColor(s_curr, cv2.COLOR_BGR2RGB),
                "prev":   cv2.cvtColor(s_prev, cv2.COLOR_BGR2RGB),
                "diff":   hot(diff),
                "binary": np.stack([binary]*3, axis=-1),
                "cands":  cands,
            }
        if fid % 1000 == 0:
            print(f"  {fid:,}/{N_SCAN:,}  ({100*fid/N_SCAN:.0f}%)  cands={len(cands)}")

    cap.release()
    scan_data = {"all_candidates": all_candidates, "sample_frames": sample_frames}
    with open(SCAN_CACHE, "wb") as fh:
        pickle.dump(scan_data, fh, protocol=4)
    print(f"Saved: {SCAN_CACHE}")

all_candidates = scan_data["all_candidates"]
sample_frames  = scan_data["sample_frames"]
n_tot  = len(all_candidates)
n_with = sum(1 for v in all_candidates.values() if v)
print(f"Frames: {n_tot:,}  |  with candidate: {n_with:,} ({100*n_with/n_tot:.1f}%)")
print(f"Sample frames cached: {len(sample_frames)}")

---
## Step 4 — Detection statistics

In [ ]:
all_scores  = [c["score"] for v in all_candidates.values() for c in v]
all_areas   = [c["area"]  for v in all_candidates.values() for c in v]
n_per_frame = [len(v) for v in all_candidates.values()]

print(f"Total candidates: {len(all_scores):,}")
if all_scores:
    print(f"Score  mean={np.mean(all_scores):.3f}  median={np.median(all_scores):.3f}  max={np.max(all_scores):.3f}")
    print(f"Area   mean={np.mean(all_areas):.1f}  max={np.max(all_areas)}")
    print(f"N/frm  mean={np.mean(n_per_frame):.2f}  max={np.max(n_per_frame)}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Detection statistics — first 5 min (full frame, no mask)", fontsize=13, fontweight="bold")

ax = axes[0,0]
ax.hist(all_scores or [0], bins=50, color="steelblue", edgecolor="white", lw=0.3)
if all_scores:
    ax.axvline(np.median(all_scores), color="red", lw=1.5, ls="--",
               label=f"median={np.median(all_scores):.3f}")
    ax.legend()
ax.set_xlabel("Score"); ax.set_ylabel("Count"); ax.set_title("Score distribution"); ax.grid(alpha=0.3)

ax = axes[0,1]
ax.hist(all_areas or [0], bins=range(MIN_AREA, MAX_AREA+2), color="darkorange", edgecolor="white", lw=0.3)
ax.axvline(15, color="red", lw=1.5, ls="--", label="ideal area=15")
ax.set_xlabel("Area (px)"); ax.set_ylabel("Count"); ax.set_title("Area distribution")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1,0]
ax.hist(n_per_frame, bins=range(0, min(30, max(n_per_frame,default=1)+2)),
        color="seagreen", edgecolor="white", lw=0.3)
ax.set_xlabel("Candidates / frame"); ax.set_ylabel("Frames")
ax.set_title("Candidates per frame"); ax.grid(alpha=0.3)

ax = axes[1,1]
if all_scores:
    sc = ax.scatter(all_areas, all_scores, s=3, alpha=0.25,
                    c=all_scores, cmap="plasma", vmin=0, vmax=max(all_scores))
    plt.colorbar(sc, ax=ax, label="score")
ax.set_xlabel("Area (px)"); ax.set_ylabel("Score")
ax.set_title("Score vs area"); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step4_histograms.png"),dpi=110,bbox_inches="tight")
plt.show()
print("Saved step4_histograms.png")

In [ ]:
fids_s   = sorted(all_candidates.keys())
best_arr = np.array([all_candidates[f][0]["score"] if all_candidates[f] else 0.0
                     for f in fids_s])
win_f    = max(1, int(ACT_WIN_S * FPS))
rate     = np.convolve((best_arr>0).astype(float), np.ones(win_f)/win_f, mode="same")
t_min    = np.array(fids_s) / FPS / 60

fig, axes = plt.subplots(2, 1, figsize=(17, 6), sharex=True)
fig.suptitle("Detection over time — first 5 min", fontsize=12, fontweight="bold")

axes[0].plot(t_min, best_arr, lw=0.5, color="steelblue", alpha=0.7)
axes[0].fill_between(t_min, 0, best_arr, alpha=0.2, color="steelblue")
axes[0].set_ylabel("Best score / frame"); axes[0].grid(alpha=0.3)
axes[0].set_title("Best candidate score per frame")

axes[1].plot(t_min, rate, lw=1.0, color="darkorange", label=f"rate ({ACT_WIN_S}s window)")
axes[1].axhline(ACT_THRESH, color="red", lw=1.2, ls="--", label=f"threshold={ACT_THRESH}")
axes[1].fill_between(t_min, 0, rate, where=rate>ACT_THRESH, alpha=0.3, color="red", label="active")
axes[1].set_ylabel("Detection rate"); axes[1].set_xlabel("Time (min)")
axes[1].set_title("Rolling detection rate"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step4_timeseries.png"),dpi=110,bbox_inches="tight")
plt.show()
print("Saved step4_timeseries.png")

---
## Step 5 — 6-panel debug frames

`original | prev frame | raw diff | binary | candidates`

Court polygon shown in cyan for reference only — not used in detection.

In [ ]:
poly_pts = COURT_POLY.reshape(-1, 2)

def draw_court(img):
    for k in range(len(poly_pts)):
        cv2.line(img, tuple(poly_pts[k]), tuple(poly_pts[(k+1)%len(poly_pts)]), (0,200,255), 1)

debug_count = 0
for fid, sf in sorted(sample_frames.items()):
    overlay = sf["orig"].copy()
    for j, c in enumerate(sf["cands"][:8]):
        col = (255,50,50) if j==0 else (255,215,0)
        r   = max(4, int((c["w"]+c["h"])/4))
        cv2.circle(overlay, (int(c["x"]),int(c["y"])), r, col, 2 if j==0 else 1)
        lbl = f"s={c['score']:.2f}" if j==0 else str(j+1)
        cv2.putText(overlay, lbl, (int(c["x"])+r+2, int(c["y"])+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.28, col, 1, cv2.LINE_AA)
    draw_court(overlay)

    t_s   = fid/FPS
    t_str = f"{int(t_s)//60}:{int(t_s)%60:02d}"
    n_c   = len(sf["cands"])
    best  = f"s={sf['cands'][0]['score']:.2f} a={sf['cands'][0]['area']}" if sf["cands"] else "none"

    fig, axes = plt.subplots(1, 5, figsize=(18, 3.2))
    for ax, (im, title) in zip(axes, [
        (sf["orig"],   f"Original  t={t_str}"),
        (sf["prev"],   f"Frame t-{DIFF_FRAMES}"),
        (sf["diff"],   "Frame diff (hot)"),
        (sf["binary"], "Binary threshold"),
        (overlay,      f"{n_c} cands  best={best}"),
    ]):
        ax.imshow(im); ax.set_title(title, fontsize=8); ax.axis("off")
    plt.suptitle(f"Frame {fid} | t={t_str} | thresh={DIFF_THRESH} | full frame", fontsize=8)
    plt.tight_layout(rect=[0,0,1,0.93])
    plt.savefig(os.path.join(DEBUG_DIR, f"debug_{fid:06d}.png"), dpi=95, bbox_inches="tight")
    plt.close()
    debug_count += 1

print(f"Saved {debug_count} debug frames -> {DEBUG_DIR}")

---
## Step 6 — Kalman filter tracker

Linear Gaussian HMM: hidden state = `[x, y, vx, vy]`.
Ball can be anywhere in frame — Kalman gates spurious detections by Mahalanobis distance.

In [ ]:
class KalmanBallTracker:
    def __init__(self, fps=25.0):
        dt = 1.0/fps
        self.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]],float)
        self.H = np.array([[1,0,0,0],[0,1,0,0]],float)
        self.Q = np.diag([KF_PROC_POS**2, KF_PROC_POS**2,
                          KF_PROC_VEL**2, KF_PROC_VEL**2])
        self.R = np.diag([KF_MEAS**2, KF_MEAS**2])
        self.x = None; self.P = None
        self.initialized = False; self.missed = 0

    def init(self, x0, y0):
        self.x = np.array([x0,y0,0.,0.])
        self.P = np.diag([100.,100.,400.,400.])
        self.initialized = True; self.missed = 0

    def predict(self):
        if not self.initialized: return None, None
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x[:2].copy(), self.P[:2,:2].copy()

    def mahalanobis(self, obs):
        y = np.array(obs) - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        return float(np.sqrt(y @ np.linalg.inv(S) @ y))

    def update(self, obs):
        z  = np.array(obs)
        y  = z - self.H @ self.x
        S  = self.H @ self.P @ self.H.T + self.R
        K  = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        self.missed = 0
        return self.x[:2].copy()

    def future(self, n):
        if not self.initialized: return []
        xp = self.x.copy()
        pts = []
        for _ in range(n):
            xp = self.F @ xp
            pts.append((float(xp[0]), float(xp[1])))
        return pts

    @property
    def speed(self):
        return float(np.linalg.norm(self.x[2:4])) if self.initialized else 0.

    @property
    def sigma(self):
        return float(np.sqrt(np.sqrt(self.P[0,0]*self.P[1,1]))) if self.initialized else np.nan

print("KalmanBallTracker ready.")

In [ ]:
kf    = KalmanBallTracker(fps=FPS)
track = {k:[] for k in ["fid","pred_x","pred_y","est_x","est_y","speed","sigma","updated"]}

for fid in sorted(all_candidates.keys()):
    cands      = all_candidates[fid]
    pred_xy, _ = kf.predict()

    if not kf.initialized:
        if cands and cands[0]["score"] > 0.3:
            kf.init(cands[0]["x"], cands[0]["y"])
        for k in ["pred_x","pred_y","est_x","est_y","speed","sigma"]:
            track[k].append(np.nan)
        track["fid"].append(fid); track["updated"].append(False)
        continue

    best, best_d = None, float("inf")
    for c in cands:
        d = kf.mahalanobis([c["x"],c["y"]])
        if d < best_d: best_d, best = d, c

    updated = best is not None and best_d < KF_GATE
    if updated:
        est_xy = kf.update([best["x"], best["y"]])
    else:
        kf.missed += 1
        if kf.missed > KF_MAX_MISS: kf.initialized = False
        est_xy = pred_xy

    track["fid"].append(fid)
    track["pred_x"].append(float(pred_xy[0]) if pred_xy is not None else np.nan)
    track["pred_y"].append(float(pred_xy[1]) if pred_xy is not None else np.nan)
    track["est_x"].append(float(est_xy[0]) if est_xy is not None else np.nan)
    track["est_y"].append(float(est_xy[1]) if est_xy is not None else np.nan)
    track["speed"].append(kf.speed); track["sigma"].append(kf.sigma)
    track["updated"].append(updated)

for k in track: track[k] = np.array(track[k])

n_upd = int(track["updated"].sum())
print(f"Kalman done.  Updates: {n_upd:,}/{len(track['fid']):,} ({100*n_upd/len(track['fid']):.1f}%)")
if track["updated"].any():
    print(f"Mean speed when updated: {np.nanmean(track['speed'][track['updated'].astype(bool)]):.2f} px/frame")

---
## Step 7 — Trajectory analysis

In [ ]:
t_m = track["fid"] / FPS / 60
val = ~np.isnan(track["est_x"])
upd = track["updated"].astype(bool)

fig, axes = plt.subplots(4, 1, figsize=(17, 12), sharex=True)
fig.suptitle("Kalman filter output — first 5 min", fontsize=13, fontweight="bold")

axes[0].plot(t_m[val], track["est_x"][val], lw=0.7, color="steelblue", label="est x")
axes[0].scatter(t_m[upd], track["est_x"][upd], s=4, color="navy", alpha=0.4, zorder=3, label="measurement")
axes[0].set_ylabel("x (px)"); axes[0].set_title("Horizontal position")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(t_m[val], track["est_y"][val], lw=0.7, color="darkorange", label="est y")
axes[1].scatter(t_m[upd], track["est_y"][upd], s=4, color="saddlebrown", alpha=0.4, zorder=3, label="measurement")
axes[1].set_ylabel("y (px)"); axes[1].set_title("Vertical position")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot(t_m[val], track["speed"][val], lw=0.7, color="seagreen")
axes[2].set_ylabel("Speed (px/frame)"); axes[2].set_title("Ball speed estimate"); axes[2].grid(alpha=0.3)

axes[3].plot(t_m[val], track["sigma"][val], lw=0.7, color="purple")
axes[3].set_ylabel("sigma (px)"); axes[3].set_xlabel("Time (min)")
axes[3].set_title("Kalman uncertainty — spikes when ball lost"); axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step7_timeseries.png"),dpi=110,bbox_inches="tight")
plt.show()
print("Saved step7_timeseries.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Ball position heatmap — Kalman estimates, first 5 min", fontsize=12, fontweight="bold")

val = ~np.isnan(track["est_x"])
xs  = track["est_x"][val]; ys = track["est_y"][val]
t_m = track["fid"] / FPS / 60

ax = axes[0]
ax.imshow(clean_bg, cmap="gray", vmin=0, vmax=255)
sc = ax.scatter(xs, ys, s=2, c=t_m[val], cmap="plasma", alpha=0.4)
poly_c = np.vstack([COURT_POLY, COURT_POLY[0]])
ax.plot(poly_c[:,0], poly_c[:,1], "c-", lw=1.5, label="court (ref)")
plt.colorbar(sc, ax=ax, label="time (min)")
ax.set_title("Positions (colour=time)"); ax.axis("off"); ax.legend(fontsize=8)

ax = axes[1]
ax.imshow(clean_bg, cmap="gray", vmin=0, vmax=255, alpha=0.5)
if len(xs):
    H, xe, ye = np.histogram2d(xs, ys,
                    bins=[np.linspace(0,FRAME_W,40), np.linspace(0,FRAME_H,23)])
    ax.imshow(H.T, extent=[0,FRAME_W,FRAME_H,0], cmap="hot", alpha=0.75, aspect="auto")
ax.plot(poly_c[:,0], poly_c[:,1], "c-", lw=1.5)
ax.set_title("Density heatmap"); ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step7_heatmap.png"),dpi=110,bbox_inches="tight")
plt.show()
print("Saved step7_heatmap.png")

In [ ]:
upd    = track["updated"].astype(bool)
val    = ~np.isnan(track["est_x"])
speeds = track["speed"][upd & val]
scores_upd = [all_candidates.get(int(f),[{}])[0].get("score",0)
              for f in track["fid"][upd & val]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Speed analysis", fontsize=12, fontweight="bold")

ax = axes[0]
ax.hist(speeds if len(speeds) else [0], bins=40, color="seagreen", edgecolor="white", lw=0.3)
if len(speeds):
    ax.axvline(np.median(speeds), color="red", lw=1.5, ls="--",
               label=f"median={np.median(speeds):.2f} px/fr")
    ax.legend()
ax.set_xlabel("Speed (px/frame)"); ax.set_ylabel("Count")
ax.set_title("Speed when ball tracked"); ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(speeds[:len(scores_upd)], scores_upd[:len(speeds)], s=4, alpha=0.3, color="steelblue")
ax.set_xlabel("Speed (px/frame)"); ax.set_ylabel("Candidate score")
ax.set_title("Speed vs score at update frames"); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step7_speed.png"),dpi=110,bbox_inches="tight")
plt.show()
print("Saved step7_speed.png")

---
## Step 8 — Activity detection

Rolling Kalman update rate -> active segments vs audio hits.

In [ ]:
win_f  = max(1, int(ACT_WIN_S * FPS))
fids_k = track["fid"].astype(int)
rate_k = np.convolve(track["updated"].astype(float), np.ones(win_f)/win_f, mode="same")
t_mk   = fids_k / FPS / 60
active = rate_k > ACT_THRESH

GROUP_GAP = 5.0
atimes = fids_k[active] / FPS
segs = []
if len(atimes):
    cs, ce = atimes[0], atimes[0]
    for t in atimes[1:]:
        if t - ce <= GROUP_GAP: ce = t
        else:
            segs.append((max(0,cs-1.0), ce+1.0)); cs=ce=t
    segs.append((max(0,cs-1.0), ce+1.0))

dur_s = float(fids_k[-1]/FPS) if len(fids_k) else 1.0
tot_s = sum(e-s for s,e in segs)
print(f"Segments: {len(segs)}  |  Active: {tot_s/60:.1f} min of {dur_s/60:.1f} min ({100*tot_s/dur_s:.1f}%)")

fig, axes = plt.subplots(3, 1, figsize=(17, 9), sharex=True)
fig.suptitle("Activity detection — video tracking vs audio hits", fontsize=12, fontweight="bold")

ax = axes[0]
ax.plot(t_mk, rate_k, lw=0.8, color="steelblue")
ax.axhline(ACT_THRESH, color="red", lw=1.2, ls="--", label=f"threshold={ACT_THRESH}")
ax.fill_between(t_mk, 0, rate_k, where=rate_k>ACT_THRESH, alpha=0.3, color="red", label="active")
ax.set_ylabel("Kalman update rate"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("Rolling Kalman update rate")

ax = axes[1]
ax.plot(t_mk, track["speed"], lw=0.5, color="seagreen", alpha=0.8)
for s,e in segs:
    ax.axvspan(s/60, e/60, alpha=0.15, color="red")
ax.set_ylabel("Speed (px/frame)"); ax.grid(alpha=0.3)
ax.set_title("Ball speed (red = active segments)")

ax = axes[2]
audio_path = str(_ROOT / "claude" / "genlabels" / "detected_thwacks_original.txt")
try:
    at = []
    with open(audio_path) as fh:
        for line in fh:
            p = line.strip().split()
            if p: at.append(float(p[0]))
    at = np.array([t for t in at if t < VID_MINUTES*60])
    ax.vlines(at/60, 0, 1, colors="steelblue", lw=0.5, alpha=0.6,
              label=f"audio hits ({len(at)} in {VID_MINUTES} min)")
    ax.fill_between(t_mk, 0, rate_k, where=rate_k>ACT_THRESH, alpha=0.3, color="red", label="video active")
    ax.legend(fontsize=8)
except FileNotFoundError:
    ax.text(0.5, 0.5, "Audio file not found (run train_thwack.ipynb first)",
            ha="center", va="center", transform=ax.transAxes)
ax.set_xlabel("Time (min)"); ax.grid(alpha=0.3)
ax.set_title("Audio hits vs video activity")

plt.tight_layout()
plt.savefig(os.path.join(GENVIDEOS_DIR,"step8_activity.png"),dpi=110,bbox_inches="tight")
plt.show()

label_out = str(_ROOT / "claude" / "genlabels" / "video_kalman_segments.txt")
os.makedirs(os.path.dirname(label_out), exist_ok=True)
with open(label_out, "w") as fh:
    for s,e in segs:
        fh.write(f"{s:.6f}\t{e:.6f}\tactive\n")
print(f"Saved {len(segs)} segments -> {label_out}")

---
## Step 9 — Generate videos (10 min)

**`framediff_10min.mp4`** — frame-diff, hot colormap, 320x180.

**`tracking_10min.mp4`** — annotated original at 640x360:
- Yellow circles = all candidates, green trail = Kalman history, orange arrow = 10-frame prediction

Takes ~3-5 min. Both written in a single pass.

In [ ]:
TRACK_HISTORY = 25
FUTURE_FRAMES = 10
OUT_SCALE     = 2
VID_MIN_VIDEO = 10

OUT_DIFF  = os.path.join(GENVIDEOS_DIR, "framediff_10min.mp4")
OUT_TRACK = os.path.join(GENVIDEOS_DIR, "tracking_10min.mp4")
OUT_W, OUT_H = FRAME_W * OUT_SCALE, FRAME_H * OUT_SCALE
N_VID = int(VID_MIN_VIDEO * 60 * FPS)

fourcc  = cv2.VideoWriter_fourcc(*"mp4v")
w_diff  = cv2.VideoWriter(OUT_DIFF,  fourcc, FPS, (FRAME_W, FRAME_H), True)
w_track = cv2.VideoWriter(OUT_TRACK, fourcc, FPS, (OUT_W, OUT_H),     True)

kf_vid   = KalmanBallTracker(fps=FPS)
pos_hist = deque(maxlen=TRACK_HISTORY)
vid_buf  = deque(maxlen=DIFF_FRAMES + 1)
poly_s   = (COURT_POLY * OUT_SCALE).reshape(-1,1,2).astype(np.int32)
s        = OUT_SCALE

cap = cv2.VideoCapture(VIDEO_PATH)
print(f"Writing {VID_MIN_VIDEO}-min videos ({N_VID:,} frames) ...")

for fid in range(N_VID):
    ok, frame = cap.read()
    if not ok: break
    small = cv2.resize(frame, (FRAME_W, FRAME_H))
    gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    vid_buf.append((gray, small))
    t_s   = fid / FPS
    t_str = f"{int(t_s)//60:02d}:{int(t_s)%60:02d}"

    if len(vid_buf) > DIFF_FRAMES:
        g_curr, s_curr = vid_buf[-1]
        g_prev, _      = vid_buf[-(DIFF_FRAMES+1)]
        diff, binary, cands = detect_candidates(g_curr, g_prev)
        diff8 = np.clip(diff.astype(float)*DIFF_AMP, 0, 255).astype(np.uint8)
        df = cv2.applyColorMap(diff8, cv2.COLORMAP_HOT)
        cv2.putText(df, t_str, (4,14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220,220,220), 1)
        w_diff.write(df)
    else:
        g_curr, s_curr = vid_buf[-1]; cands = []
        w_diff.write(np.zeros((FRAME_H, FRAME_W, 3), np.uint8))

    # Kalman
    pred_xy, _ = kf_vid.predict()
    if not kf_vid.initialized:
        if cands and cands[0]["score"] > 0.3:
            kf_vid.init(cands[0]["x"], cands[0]["y"])
    else:
        best, best_d = None, float("inf")
        for c in cands:
            d = kf_vid.mahalanobis([c["x"],c["y"]])
            if d < best_d: best_d, best = d, c
        if best is not None and best_d < KF_GATE:
            kf_vid.update([best["x"],best["y"]])
            pos_hist.append((kf_vid.x[0], kf_vid.x[1]))
        else:
            kf_vid.missed += 1
            if kf_vid.missed > KF_MAX_MISS:
                kf_vid.initialized = False; pos_hist.clear()

    # tracking frame
    draw = cv2.resize(s_curr, (OUT_W, OUT_H))
    cv2.polylines(draw, [poly_s], True, (0,200,255), 1)   # court ref only

    for c in cands:
        cx, cy = int(c["x"]*s), int(c["y"]*s)
        r = max(3, int((c["w"]+c["h"])/4*s))
        cv2.circle(draw, (cx,cy), r, (0,215,255), 1)
        cv2.putText(draw, f"{c['score']:.2f}", (cx+r+2,cy+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.22, (0,215,255), 1)

    for i, (px,py) in enumerate(list(pos_hist)):
        fade = (i+1)/max(len(pos_hist),1)
        cv2.circle(draw, (int(px*s),int(py*s)), max(1,int(5*fade)), (0,int(220*fade),0), -1)

    if kf_vid.initialized:
        fut = kf_vid.future(FUTURE_FRAMES)
        for j in range(len(fut)-1):
            fade = 1.0 - j/len(fut)
            cv2.line(draw,
                     (int(fut[j][0]*s),int(fut[j][1]*s)),
                     (int(fut[j+1][0]*s),int(fut[j+1][1]*s)),
                     (0,int(80*fade),int(255*fade)), 2)
        if len(fut) >= 2:
            cv2.arrowedLine(draw,
                            (int(fut[-2][0]*s),int(fut[-2][1]*s)),
                            (int(fut[-1][0]*s),int(fut[-1][1]*s)),
                            (0,30,255), 2, tipLength=0.4)
        cv2.circle(draw, (int(kf_vid.x[0]*s),int(kf_vid.x[1]*s)), 6, (0,255,80), 2)

    cv2.putText(draw, t_str, (4,18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
    cv2.putText(draw, "YELLOW=candidates  GREEN=track  ORANGE=prediction",
                (4, OUT_H-6), cv2.FONT_HERSHEY_SIMPLEX, 0.28, (200,200,200), 1)
    w_track.write(draw)

    if fid % 2000 == 0:
        print(f"  {fid:,}/{N_VID:,}  ({100*fid/N_VID:.0f}%)")

cap.release(); w_diff.release(); w_track.release()
print("Done!")
for p in [OUT_DIFF, OUT_TRACK]:
    mb = os.path.getsize(p)/1e6 if os.path.exists(p) else 0
    print(f"  {p}  ({mb:.1f} MB)")